In [ ]:
# prompt: 드라이브 마운트 코드

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install geokakao

In [ ]:
import folium
import geokakao as gk
import pandas as pd

In [ ]:
df_subway = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/subway_line_1_8_20231231.csv')
df_addr = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/seoul_subway_address_2023.csv')
print(df_subway)


            연번        수송일자  호선명   역번호   역명 승하차구분 승객유형  t06시간대이전  t06_07시간대  \
0            1  2023-07-01    1   150  서울역    승차   일반       192        192   
1            2  2023-07-01    1   150  서울역    승차  어린이         0          1   
2            3  2023-07-01    1   150  서울역    승차  중고생         0          0   
3            4  2023-07-01    1   150  서울역    승차  청소년         0          3   
4            5  2023-07-01    1   150  서울역    승차  우대권        88        105   
...        ...         ...  ...   ...  ...   ...  ...       ...        ...   
817639  817642  2023-12-31    8  2828  남위례    하차   일반         7         37   
817640  817643  2023-12-31    8  2828  남위례    하차  어린이         0          0   
817641  817644  2023-12-31    8  2828  남위례    하차  청소년         0          0   
817642  817645  2023-12-31    8  2828  남위례    하차  우대권        10         15   
817643  817646  2023-12-31    8  2828  남위례    하차   직원         2          1   

        t07_08시간대  ...  t15_16시간대  t16_17시간대  t17_18시간대  t18_19

In [ ]:
print(df_addr)

      연번   역번호  호선           역명         역전화번호                       도로명주소  \
0      1   150   1           서울  02-6110-1331  서울특별시 중구 세종대로 지하2(남대문로 5가)   
1      2   151   1           시청  02-6110-1321     서울특별시 중구 세종대로 지하101(정동)   
2      3   152   1           종각  02-6110-1311     서울특별시 종로구 종로 지하55(종로1가)   
3      4   153   1         종로3가  02-6110-1301    서울특별시 종로구 종로 지하129(종로3가)   
4      5   154   1         종로5가  02-6110-1291    서울특별시 종로구 종로 지하216(종로5가)   
..   ...   ...  ..          ...           ...                         ...   
283  284  4134   9         송파나루  02-2656-0934  서울특별시 송파구 백제고분로 지하446(방이동)   
284  285  4135   9         한성백제  02-2656-0935   서울특별시 송파구 위례성대로 지하29(방이동)   
285  286  4136   9  올림픽공원(한국체대)  02-2656-0936  서울특별시 송파구 양재대로 지하1233(방이동)   
286  287  4137   9         둔촌오륜  02-2656-0937   서울특별시 강동구 강동대로 지하303(둔촌동)   
287  288  4138   9       중앙보훈병원  02-2656-0938    서울특별시 강동구 동남로 지하625(둔촌동)   

                                지번주소  
0      서울특별시 중구 남대문로5가 73-6 서울역(1호선)

In [ ]:
time_zone = ['t07_08시간대', 't08_09시간대']
df_subway = df_subway.loc[df_subway.호선명 == 2]  # 2호선 추출
df_subway = df_subway.groupby('역명')[time_zone].sum()  # 역별 승객 수 집계
df_subway['탑승객 수'] = df_subway.sum(axis=1)  # 탑승객 수 합계
df_subway.head()

,t07_08시간대,t08_09시간대,탑승객 수
역명,,,
강남,1113322,2136584,3249906
강변(동서울터미널),603998,943006,1547004
건대입구,464041,915444,1379485
교대(법원.검찰청),482659,1125726,1608385
구로디지털단지,1325635,2397609,3723244


In [ ]:
df_merge = pd.merge(df_subway, df_addr, on='역명', how='inner') #inner가 기본
df_merge.head()

,역명,t07_08시간대,t08_09시간대,탑승객 수,연번,역번호,호선,역전화번호,도로명주소,지번주소
0,강남,1113322,2136584,3249906,32,222,2,02-6110-2221,서울특별시 강남구 강남대로 지하396(역삼동),서울특별시 강남구 역삼동 858 강남역(2호선)
1,강변(동서울터미널),603998,943006,1547004,24,214,2,02-6110-2141,서울특별시 광진구 강변역로 53(구의동),서울특별시 광진구 구의동 546-6 강변역(2호선)
2,건대입구,464041,915444,1379485,22,212,2,02-6110-2121,서울특별시 광진구 아차산로 243(화양동),서울특별시 광진구 화양동 7-3 2호선 건대입구역(2호선)
3,건대입구,464041,915444,1379485,234,2729,7,02-6311-7271,서울특별시 광진구 능동로 지하110(화양동),서울특별시 광진구 화양동 6-3 건대입구역(7호선)
4,구로디지털단지,1325635,2397609,3723244,42,232,2,02-6110-2321,서울특별시 구로구 도림천로 477(구로동),서울특별시 구로구 구로동 810-3 구로디지털단지역(2호선)


In [ ]:
# add lat, lon
gk.add_coordinates_to_dataframe(df_merge, '도로명주소')

# 문자열 좌푯값을 숫자로 변환
df_merge.decimalLatitude = pd.to_numeric(df_merge.decimalLatitude)
df_merge.decimalLongitude = pd.to_numeric(df_merge.decimalLongitude)
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   역명                64 non-null     object 
 1   t07_08시간대         64 non-null     int64  
 2   t08_09시간대         64 non-null     int64  
 3   탑승객 수             64 non-null     int64  
 4   연번                64 non-null     int64  
 5   역번호               64 non-null     int64  
 6   호선                64 non-null     int64  
 7   역전화번호             64 non-null     object 
 8   도로명주소             64 non-null     object 
 9   지번주소              64 non-null     object 
 10  decimalLatitude   64 non-null     float64
 11  decimalLongitude  64 non-null     float64
dtypes: float64(2), int64(6), object(4)
memory usage: 6.1+ KB


In [ ]:
center = df_merge[['decimalLatitude', 'decimalLongitude']].mean().to_list()

In [ ]:
# 지도 객체 생성
map = folium.Map(location=center, zoom_start=12)

In [ ]:
# 지도에 마커 추가
for i in range(len(df_merge)):
    folium.Marker(location=[df_merge.loc[i, 'decimalLatitude'],
                            df_merge.loc[i, 'decimalLongitude']],
                            icon=folium.Icon(color='red', icon='star')).add_to(map)

In [ ]:
map

In [ ]:
# 지도 객체 생성
map = folium.Map(location=center, zoom_start=12)

for i in range(len(df_merge)):
    folium.CircleMarker(location=[df_merge.loc[i, 'decimalLatitude'],
    df_merge.loc[i, 'decimalLongitude']],
    radius=((df_merge.loc[i, '탑승객 수']/150000)),  # 원의 반지름
    color='red',  # 원의 색
    stroke=False,  # 윤곽선 없음
    fill=True,  # 원의 내부 색
    fill_opacity='50%'  # 원의 내부 색 투명도
    ).add_to(map)

In [ ]:
map

In [ ]:
# 지도에 텍스트(역명) 추가
html_start = html = '<div \
style="\
font-size: 12px;\
color: blue;\
background-color:rgba(255, 255, 255, 0.2);\
width:85px;\
text-align:left;\
margin:0px;\
"><b>'
html_end = '</b></div>'

for i in range(len(df_merge)):
    folium.Marker(location=[df_merge.loc[i, 'decimalLatitude'],
    df_merge.loc[i, 'decimalLongitude']],
    icon=folium.DivIcon(
        icon_anchor=(0, 0),  # 텍스트 위치 설정
        html=html_start+df_merge.loc[i, '역명']+html_end
        )).add_to(map)


In [ ]:
map

1. 출근 시간대 탑승객 수가 가장 많은 역 Top 5 구하고 지도에 출력

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd
import folium
import geokakao as gk

# ----------------------------------
# 1. 데이터 불러오기
# ----------------------------------
df_subway = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/subway_line_1_8_20231231.csv')
df_addr = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/seoul_subway_address_2023.csv')

# ----------------------------------
# 2. 사용할 시간대 설정
#    오전 7~9시
# ----------------------------------
time_zone = ['t07_08시간대', 't08_09시간대']

# ----------------------------------
# 3. 데이터 형식 통일
#    호선명을 숫자로 맞추기
# ----------------------------------
df_subway['호선명'] = pd.to_numeric(df_subway['호선명'], errors='coerce')
df_addr['호선'] = pd.to_numeric(df_addr['호선'], errors='coerce')

# ----------------------------------
# 4. 노선별 + 역별 오전 7~9시 승객 수 집계
# ----------------------------------
df_grouped = df_subway.groupby(['호선명', '역명'])[time_zone].sum().reset_index()

# 오전 7~9시 총합 컬럼 만들기
df_grouped['탑승객 수'] = df_grouped[time_zone].sum(axis=1)

# 확인
print("노선별 역별 집계 결과")
print(df_grouped.head())

# ----------------------------------
# 5. 각 노선별 탑승객 수 상위 5개 역 추출
# ----------------------------------
df_top5 = (
    df_grouped.sort_values(['호선명', '탑승객 수'], ascending=[True, False])
              .groupby('호선명')
              .head(5)
              .reset_index(drop=True)
)

print("\n노선별 상위 5개 역")
print(df_top5)

# ----------------------------------
# 6. 주소 데이터의 컬럼명을 맞추기
#    df_addr의 '호선' -> '호선명'
# ----------------------------------
df_addr = df_addr.rename(columns={'호선': '호선명'})

# ----------------------------------
# 7. 노선명 + 역명 기준으로 merge
#    환승역 중복 문제 방지
# ----------------------------------
df_merge = pd.merge(df_top5, df_addr, on=['호선명', '역명'], how='inner')

print("\n주소 결합 후 결과")
print(df_merge[['호선명', '역명', '도로명주소']])

# ----------------------------------
# 8. 주소를 위도, 경도로 변환
# ----------------------------------
gk.add_coordinates_to_dataframe(df_merge, '도로명주소')

# 좌표를 숫자형으로 변환
df_merge['decimalLatitude'] = pd.to_numeric(df_merge['decimalLatitude'], errors='coerce')
df_merge['decimalLongitude'] = pd.to_numeric(df_merge['decimalLongitude'], errors='coerce')

# 좌표 없는 행 제거
df_merge = df_merge.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print("\n좌표 추가 후 결과")
print(df_merge[['호선명', '역명', 'decimalLatitude', 'decimalLongitude']])

# ----------------------------------
# 9. 지도 중심점 계산
# ----------------------------------
center = df_merge[['decimalLatitude', 'decimalLongitude']].mean().to_list()

# 지도 객체 생성
map = folium.Map(location=center, zoom_start=11)

# ----------------------------------
# 10. 노선별 색상 지정
# ----------------------------------
line_colors = {
    1: 'red',
    2: 'green',
    3: 'orange',
    4: 'blue',
    5: 'purple',
    6: 'darkred',
    7: 'darkgreen',
    8: 'cadetblue'
}
# ----------------------------------
# 11. 지도에 원 추가
#     원 크기 = 탑승객 수 비례
#     팝업 = 노선명, 역명, 탑승객 수
# ----------------------------------
for i in range(len(df_merge)):
    line = df_merge.loc[i, '호선명']
    station = df_merge.loc[i, '역명']
    passengers = df_merge.loc[i, '탑승객 수']
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']

    color = line_colors.get(line, 'red')

    folium.CircleMarker(
        location=[lat, lon],
        radius=passengers / 200000,   # 원 크기 조절
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.6,
        popup=f"{line}호선 / {station} / {passengers:,}명"
    ).add_to(map)

# ----------------------------------
# 12. 역 이름 텍스트 추가
# ----------------------------------
html_start = '''
<div style="
font-size:12px;
color:blue;
background-color:rgba(255,255,255,0.3);
width:90px;
text-align:left;
margin:0px;
"><b>
'''
html_end = '</b></div>'

for i in range(len(df_merge)):
    station = df_merge.loc[i, '역명']
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            icon_anchor=(0, 0),
            html=html_start + station + html_end
        )
    ).add_to(map)

# ----------------------------------
# 13. 지도 출력
# ----------------------------------
map

노선별 역별 집계 결과
   호선명   역명  t07_08시간대  t08_09시간대    탑승객 수
0    1  동대문     156348     237374   393722
1    1  동묘앞      98160     192553   290713
2    1  서울역     916962    1794997  2711959
3    1   시청     472859    1209611  1682470
4    1  신설동     264842     513925   778767

노선별 상위 5개 역
    호선명                역명  t07_08시간대  t08_09시간대    탑승객 수
0     1               서울역     916962    1794997  2711959
1     1                종각     686784    1638831  2325615
2     1                시청     472859    1209611  1682470
3     1              종로5가     286309     641223   927532
4     1      청량리(서울시립대입구)     370926     541695   912621
5     2           구로디지털단지    1325635    2397609  3723244
6     2                강남    1113322    2136584  3249906
7     2                역삼     975506    2273981  3249487
8     2          잠실(송파구청)    1183539    2046564  3230103
9     2                신림    1257579    1843799  3101378
10    3          양재(서초구청)     828788    1621007  2449795
11    3               연신내     84

지하철 1호선 퇴근시간 승객 정보

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd
import folium
import geokakao as gk

# ----------------------------------
# 1. 데이터 불러오기
# ----------------------------------
df_subway = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/subway_line_1_8_20231231.csv')
df_addr = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/seoul_subway_address_2023.csv')

# ----------------------------------
# 2. 1호선만 선택
# ----------------------------------
df_subway['호선명'] = pd.to_numeric(df_subway['호선명'], errors='coerce')
df_addr['호선'] = pd.to_numeric(df_addr['호선'], errors='coerce')

df_subway_line1 = df_subway[df_subway['호선명'] == 1].copy()
df_addr_line1 = df_addr[df_addr['호선'] == 1].copy()

# ----------------------------------
# 3. 퇴근 시간대 설정
# ----------------------------------
evening_cols = ['t18_19시간대', 't19_20시간대']

# ----------------------------------
# 4. 1호선 역별 퇴근 시간대 승객 수 집계
# ----------------------------------
df_grouped = df_subway_line1.groupby(['호선명', '역명'])[evening_cols].sum().reset_index()

# 퇴근 시간대 총합 계산
df_grouped['퇴근시간 승객 수'] = df_grouped[evening_cols].sum(axis=1)

print("1호선 역별 퇴근 시간대 승객 수")
print(df_grouped[['역명', '퇴근시간 승객 수']])

# ----------------------------------
# 5. 주소 데이터 컬럼명 맞추기
# ----------------------------------
df_addr_line1 = df_addr_line1.rename(columns={'호선': '호선명'})

# ----------------------------------
# 6. 노선명 + 역명 기준으로 결합
# ----------------------------------
df_merge = pd.merge(df_grouped, df_addr_line1, on=['호선명', '역명'], how='inner')

print("\n주소 결합 후 결과")
print(df_merge[['호선명', '역명', '도로명주소']])

# ----------------------------------
# 7. 주소를 위도/경도로 변환
# ----------------------------------
gk.add_coordinates_to_dataframe(df_merge, '도로명주소')

df_merge['decimalLatitude'] = pd.to_numeric(df_merge['decimalLatitude'], errors='coerce')
df_merge['decimalLongitude'] = pd.to_numeric(df_merge['decimalLongitude'], errors='coerce')

df_merge = df_merge.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print("\n좌표 추가 후 결과")
print(df_merge[['역명', 'decimalLatitude', 'decimalLongitude']])

# ----------------------------------
# 8. 지도 중심 설정
# ----------------------------------
center = df_merge[['decimalLatitude', 'decimalLongitude']].mean().to_list()
map = folium.Map(location=center, zoom_start=11)

# ----------------------------------
# 9. 1호선 모든 역을 원으로 표시
#    원 크기 = 퇴근 시간 승객 수
# ----------------------------------
for i in range(len(df_merge)):
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']
    station = df_merge.loc[i, '역명']
    evening_passengers = df_merge.loc[i, '퇴근시간 승객 수']

    folium.CircleMarker(
        location=[lat, lon],
        radius=max(evening_passengers / 50000, 5),   # 원 크기 조절
        color='black',
        weight=1,
        fill=True,
        fill_color='red',
        fill_opacity=0.6,
        popup=f"1호선 / {station}<br>퇴근시간 승객 수: {evening_passengers:,}명"
    ).add_to(map)

# ----------------------------------
# 10. 역명 텍스트 추가
# ----------------------------------
html_start = '''
<div style="
font-size:12px;
color:darkblue;
background-color:rgba(255,255,255,0.4);
width:85px;
text-align:left;
margin:0px;
"><b>
'''
html_end = '</b></div>'

for i in range(len(df_merge)):
    station = df_merge.loc[i, '역명']
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            icon_anchor=(0, 0),
            html=html_start + station + html_end
        )
    ).add_to(map)

# ----------------------------------
# 11. 지도 출력
# ----------------------------------
map

1호선 역별 퇴근 시간대 승객 수
             역명  퇴근시간 승객 수
0           동대문     505106
1           동묘앞     304832
2           서울역    3023257
3            시청    1621377
4           신설동     743534
5           제기동     625826
6            종각    2364906
7          종로3가    1305772
8          종로5가    1218678
9  청량리(서울시립대입구)     928077

주소 결합 후 결과
   호선명            역명                      도로명주소
0    1           동대문    서울특별시 종로구 종로 지하302(창신동)
1    1           동묘앞      서울특별시 종로구 종로 359(숭인동)
2    1            시청    서울특별시 중구 세종대로 지하101(정동)
3    1           신설동    서울특별시 동대문구 왕산로 지하1(신설동)
4    1           제기동   서울특별시 동대문구 왕산로 지하93(제기동)
5    1            종각    서울특별시 종로구 종로 지하55(종로1가)
6    1          종로3가   서울특별시 종로구 종로 지하129(종로3가)
7    1          종로5가   서울특별시 종로구 종로 지하216(종로5가)
8    1  청량리(서울시립대입구)  서울특별시 동대문구 왕산로 지하205(전농동)

좌표 추가 후 결과
             역명  decimalLatitude  decimalLongitude
0           동대문        37.571778        127.011167
1           동묘앞        37.573372        127.016751
2            시청        37.5

지하철 1호선 출근 시간대와 퇴근 시간대 승객 차이 정보

In [ ]:
# 필요한 라이브러리 불러오기
import pandas as pd
import folium
import geokakao as gk

# ----------------------------------
# 1. 데이터 불러오기
# ----------------------------------
df_subway = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/subway_line_1_8_20231231.csv')
df_addr = pd.read_csv('/content/drive/MyDrive/MyLecture/2025/데이터시각화/지도시각화/seoul_subway_address_2023.csv')

# ----------------------------------
# 2. 1호선만 선택
# ----------------------------------
df_subway['호선명'] = pd.to_numeric(df_subway['호선명'], errors='coerce')
df_addr['호선'] = pd.to_numeric(df_addr['호선'], errors='coerce')

df_subway_line1 = df_subway[df_subway['호선명'] == 1].copy()
df_addr_line1 = df_addr[df_addr['호선'] == 1].copy()

# ----------------------------------
# 3. 출근 / 퇴근 시간대 설정
# ----------------------------------
morning_cols = ['t07_08시간대', 't08_09시간대']   # 출근 시간대
evening_cols = ['t18_19시간대', 't19_20시간대']   # 퇴근 시간대

# ----------------------------------
# 4. 1호선 역별 집계
# ----------------------------------
df_grouped = df_subway_line1.groupby(['호선명', '역명'])[morning_cols + evening_cols].sum().reset_index()

# 출근 / 퇴근 시간대 합계
df_grouped['출근시간대'] = df_grouped[morning_cols].sum(axis=1)
df_grouped['퇴근시간대'] = df_grouped[evening_cols].sum(axis=1)

# 차이 계산
df_grouped['차이'] = abs(df_grouped['출근시간대'] - df_grouped['퇴근시간대'])

print("1호선 역별 집계 결과")
print(df_grouped[['역명', '출근시간대', '퇴근시간대', '차이']].head())

# ----------------------------------
# 5. 주소 데이터 컬럼명 맞추기
# ----------------------------------
df_addr_line1 = df_addr_line1.rename(columns={'호선': '호선명'})

# ----------------------------------
# 6. 노선명 + 역명 기준으로 결합
# ----------------------------------
df_merge = pd.merge(df_grouped, df_addr_line1, on=['호선명', '역명'], how='inner')

print("\n주소 결합 후 결과")
print(df_merge[['호선명', '역명', '도로명주소']].head())

# ----------------------------------
# 7. 주소를 위도/경도로 변환
# ----------------------------------
gk.add_coordinates_to_dataframe(df_merge, '도로명주소')

df_merge['decimalLatitude'] = pd.to_numeric(df_merge['decimalLatitude'], errors='coerce')
df_merge['decimalLongitude'] = pd.to_numeric(df_merge['decimalLongitude'], errors='coerce')

df_merge = df_merge.dropna(subset=['decimalLatitude', 'decimalLongitude'])

print("\n좌표 추가 후 결과")
print(df_merge[['역명', 'decimalLatitude', 'decimalLongitude']].head())

# ----------------------------------
# 8. 지도 중심 설정
# ----------------------------------
center = df_merge[['decimalLatitude', 'decimalLongitude']].mean().to_list()
map = folium.Map(location=center, zoom_start=11)

# ----------------------------------
# 9. 1호선 모든 역 기본 마커 표시
# ----------------------------------
for i in range(len(df_merge)):
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']
    station = df_merge.loc[i, '역명']

    folium.Marker(
        location=[lat, lon],
        popup=f"1호선 / {station}",
        icon=folium.Icon(color='gray', icon='info-sign')
    ).add_to(map)

# ----------------------------------
# 10. 출근형 / 퇴근형 색상 구분
# ----------------------------------
def get_color(row):
    if row['출근시간대'] > row['퇴근시간대']:
        return 'blue'   # 출근형
    else:
        return 'red'    # 퇴근형

# ----------------------------------
# 11. 차이를 원 크기로 표시
# ----------------------------------
for i in range(len(df_merge)):
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']
    station = df_merge.loc[i, '역명']
    morning = df_merge.loc[i, '출근시간대']
    evening = df_merge.loc[i, '퇴근시간대']
    diff = df_merge.loc[i, '차이']

    color = get_color(df_merge.loc[i])

    folium.CircleMarker(
        location=[lat, lon],
        radius=diff / 20000,   # 차이에 비례한 크기
        color='black',
        weight=1,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=(
            f"1호선 / {station}<br>"
            f"출근시간대: {morning:,}명<br>"
            f"퇴근시간대: {evening:,}명<br>"
            f"차이: {diff:,}명"
        )
    ).add_to(map)

# ----------------------------------
# 12. 역명 텍스트 추가
# ----------------------------------
html_start = '''
<div style="
font-size:12px;
color:darkblue;
background-color:rgba(255,255,255,0.4);
width:85px;
text-align:left;
margin:0px;
"><b>
'''
html_end = '</b></div>'

for i in range(len(df_merge)):
    station = df_merge.loc[i, '역명']
    lat = df_merge.loc[i, 'decimalLatitude']
    lon = df_merge.loc[i, 'decimalLongitude']

    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            icon_anchor=(0, 0),
            html=html_start + station + html_end
        )
    ).add_to(map)

# ----------------------------------
# 13. 지도 출력
# ----------------------------------
map

1호선 역별 집계 결과
    역명    출근시간대    퇴근시간대      차이
0  동대문   393722   505106  111384
1  동묘앞   290713   304832   14119
2  서울역  2711959  3023257  311298
3   시청  1682470  1621377   61093
4  신설동   778767   743534   35233

주소 결합 후 결과
   호선명   역명                     도로명주소
0    1  동대문   서울특별시 종로구 종로 지하302(창신동)
1    1  동묘앞     서울특별시 종로구 종로 359(숭인동)
2    1   시청   서울특별시 중구 세종대로 지하101(정동)
3    1  신설동   서울특별시 동대문구 왕산로 지하1(신설동)
4    1  제기동  서울특별시 동대문구 왕산로 지하93(제기동)

좌표 추가 후 결과
    역명  decimalLatitude  decimalLongitude
0  동대문        37.571778        127.011167
1  동묘앞        37.573372        127.016751
2   시청        37.565439        126.976983
3  신설동        37.574719        127.025094
4  제기동        37.578197        127.034691
